# 02.01 Data Quality and Verification
**Module:** Data Preparation & Quality Assurance  
**Target Dataset:** `data/synthetic/tabular/ofphil_synthetic_raw.csv`  
**Expected Audit Artifact:** `data/interim/data_quality/missingness_profile.csv`

---
### Research Objectives
1. **Schema & Dimensionality Audit:** Inspect column inventory (59 columns), data types, and record volume (500 records).
2. **Missingness & Completeness Profiling:** Calculate exact missing value frequencies and percentages per column, exporting the missingness profile artifact.
3. **Duplicate & Inconsistency Detection:** Detect exact row duplicates and quasi-identifier collisions.
4. **Domain Range & Outlier Validation:** Validate clinical and socioeconomic invariants (age, household size, family income).
5. **Categorical Domain Verification:** Audit distinct values and cardinality of primary categorical indicators.
6. **Demographic Representation & Fairness Audit:** Analyze distribution across protected attributes (`sex`, `age_group`), assess subgroup representations, and document the baseline demographic parity difference (0.2302 across age groups) as reported in the generation report. Reinforce that balanced target classes (50/50) do not imply bias-free demographic representation.

### Inputs & Expected Outputs
* **Inputs:**
  * `data/synthetic/tabular/ofphil_synthetic_raw.csv` (Raw synthetic tabular dataset)
* **Outputs:**
  * `data/interim/data_quality/missingness_profile.csv` (Detailed missingness profiling artifact)

### 1. Environment Setup & Library Imports
Initialize project paths, standard scientific libraries (`pandas`, `numpy`, `pathlib`), and project root resolver.

In [1]:
from pathlib import Path
import json
import datetime
import re
import pandas as pd
import numpy as np

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
print(f"Project root resolved to: {PROJECT_ROOT}")

Project root resolved to: c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2


### 2. Load Raw Synthetic Dataset & Inspect Schema
Load `ofphil_synthetic_raw.csv` and verify dimensions (expected 500 rows × 59 columns).

In [2]:
SYNTH_CSV = PROJECT_ROOT / 'data' / 'synthetic' / 'tabular' / 'ofphil_synthetic_raw.csv'
AUDIT_CSV_PATH = PROJECT_ROOT / 'data' / 'interim' / 'data_quality' / 'missingness_profile.csv'
AUDIT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SYNTH_CSV)
print(f"Dataset Loaded: {len(df)} records x {df.shape[1]} columns from {SYNTH_CSV.name}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"\nFirst 5 Columns: {list(df.columns[:5])}")
print(f"Last 5 Columns: {list(df.columns[-5:])}")

Dataset Loaded: 500 records x 59 columns from ofphil_synthetic_raw.csv
Memory Usage: 1031.89 KB

First 5 Columns: ['age', 'sex', 'marital_status', 'number_of_dependents', 'monthly_family_income']
Last 5 Columns: ['document_category_diversity', 'date_available', 'date_registered', 'status', 'eligibility']


### 3. Completeness & Missing Value Profiling
Calculate null count, non-null count, and percentage missing for every column. Export artifact to `data/interim/data_quality/missingness_profile.csv`.

In [3]:
profile_rows = []
for col in df.columns:
    null_cnt = int(df[col].isna().sum())
    non_null_cnt = int(len(df) - null_cnt)
    null_pct = round((null_cnt / len(df)) * 100.0, 2)
    profile_rows.append({
        'column_name': col,
        'dtype': str(df[col].dtype),
        'null_count': null_cnt,
        'null_percentage': null_pct,
        'non_null_count': non_null_cnt,
        'source_dataset': SYNTH_CSV.name,
    })

profile_df = pd.DataFrame(profile_rows)
profile_df.to_csv(AUDIT_CSV_PATH, index=False)
print(f"Missingness profile exported -> {AUDIT_CSV_PATH} ({len(profile_df)} columns)")

# Display columns with highest missingness
top_missing = profile_df.sort_values(by='null_count', ascending=False).head(10)
print("\nTop Columns by Missing Values:")
print(top_missing[['column_name', 'dtype', 'null_count', 'null_percentage']].to_string(index=False))

Missingness profile exported -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\data_quality\missingness_profile.csv (59 columns)

Top Columns by Missing Values:
                            column_name   dtype  null_count  null_percentage
       other_eligibility_certifications float64         500            100.0
          existing_financial_assistance     str         371             74.2
                         follow_up_date     str         322             64.4
                    sectoral_membership     str         248             49.6
social_welfare_assessment_certification     str         219             43.8
              outstanding_medical_bills     str         219             43.8
                 educational_attainment     str         217             43.4
   ngo_charity_assistance_certification     str         208             41.6
                      disability_status     str         207             41.4
    government_ass

### 4. Duplicate & Quasi-Identifier Collision Detection
Verify exact duplicate rows and examine potential collision on quasi-identifying demographic combinations (`age`, `sex`, `marital_status`, `occupation`).

In [4]:
exact_dups = int(df.duplicated().sum())
print(f"Exact Duplicate Rows: {exact_dups}")

quasi_cols = ['age', 'sex', 'marital_status', 'occupation']
available_quasi = [c for c in quasi_cols if c in df.columns]
if available_quasi:
    quasi_dup_count = int(df.duplicated(subset=available_quasi).sum())
    print(f"Quasi-Identifier ({', '.join(available_quasi)}) Duplicate Clusters: {quasi_dup_count}")

Exact Duplicate Rows: 0
Quasi-Identifier (age, sex, marital_status, occupation) Duplicate Clusters: 78


### 5. Numerical Range & Categorical Value Invariant Validation
Perform domain validation on numeric variables and audit distinct values for core categorical indicators.

In [5]:
# 1. Numerical domain & relational invariants
age_numeric = pd.to_numeric(df['age'], errors='coerce')
invalid_age = int(((age_numeric < 0) | (age_numeric > 120)).sum())
null_age = int(age_numeric.isna().sum())

hs_numeric = pd.to_numeric(df['household_size'], errors='coerce')
invalid_hs = int((hs_numeric < 1).sum())

dep_numeric = pd.to_numeric(df['number_of_dependents'], errors='coerce')
invalid_dep = int((dep_numeric < 0).sum())
dep_ge_hs = int((dep_numeric >= hs_numeric).sum())

def _parse_php(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r'[\d,]+(?:\.\d+)?', str(val))
    return float(m.group(0).replace(',', '')) if m else np.nan

inc_num = df['monthly_family_income'].apply(_parse_php)
obl_num = df['monthly_household_obligations_expenses'].apply(_parse_php)
util_num = df['monthly_utility_expenses'].apply(_parse_php)
util_gt_obl = int((util_num > obl_num).sum())

print("Numerical & Relational Domain Validation:")
print(f" - Age: {invalid_age} invalid values (<0 or >120), {null_age} nulls")
print(f" - Household Size: {invalid_hs} invalid values (<1)")
print(f" - Dependents: {invalid_dep} negative values, {dep_ge_hs} rows where dependents >= household_size")
print(f" - Utility > Household Obligations (flagged for review): {util_gt_obl} rows")
print(f" - 100% Null Structural Columns: {[c for c in df.columns if df[c].isna().all()]}")

# 2. Categorical cardinalities
cat_audit_cols = ['sex', 'marital_status', 'employment_status', 'residence_status', 'housing_condition', 'diagnosis_severity', 'eligibility']
print("\nCategorical Cardinality & Unique Values:")
for c in cat_audit_cols:
    if c in df.columns:
        uniques = df[c].dropna().unique().tolist()
        print(f" - {c} ({len(uniques)} distinct): {uniques}")

Numerical & Relational Domain Validation:
 - Age: 0 invalid values (<0 or >120), 0 nulls
 - Household Size: 0 invalid values (<1)
 - Dependents: 0 negative values, 0 rows where dependents >= household_size
 - Utility > Household Obligations (flagged for review): 6 rows
 - 100% Null Structural Columns: ['other_eligibility_certifications']

Categorical Cardinality & Unique Values:
 - sex (2 distinct): ['F', 'M']
 - marital_status (7 distinct): ['Married', 'Single', 'Separated', 'Common-law partner', 'Live-in', 'Widowed', 'Common-Law']
 - employment_status (6 distinct): ['Self-Employed', 'Employed', 'Student', 'Unemployed', 'Retired', 'Pensioner']
 - residence_status (15 distinct): ['Owner', 'Renter', 'Owned', 'Informal Settler', 'Rent-free', 'Living with parents', 'Free (Owned by Sibling)', 'Living with daughter', 'Renting', 'Mortgaged', 'Owned by aunt', 'Rented', 'Owner (Mother-in-Law)', 'Owner (Siblings)', 'Living with niece']
 - housing_condition (23 distinct): ['Concrete', 'Living wi

### 6. Demographic Representation, Fairness & Proxy Variable Audit
**Fairness Context:**
- The target distribution is strictly balanced (250 ELIGIBLE vs 250 INELIGIBLE).
- However, **balanced target classes do NOT imply a bias-free dataset**.
- Generation report identifies a baseline **Demographic Parity Difference of 0.2302 across age groups**.
- Direct protected attributes (`sex`, `age`) must NOT be used as eligibility determinants.
- We must audit potential proxy variables (`occupation`, `residence_status`, `educational_attainment`) that may correlate with protected demographics.

In [ ]:
# Target class distribution
print("Target Distribution:")
print(df['eligibility'].value_counts(dropna=False).to_frame('count'))

# Sex representation
print("\nDemographic Distribution - Sex:")
sex_counts = df['sex'].value_counts(dropna=False)
print(sex_counts.to_frame('count').assign(percentage=lambda x: round(x['count'] / len(df) * 100, 2)))

# Age group representation (computed as a standalone series so df remains strictly 500 x 59)
bins = [0, 18, 40, 60, 120]
labels = ['Pediatric (<18)', 'Young Adult (18-39)', 'Middle Age (40-59)', 'Senior (60+)']
age_group = pd.cut(pd.to_numeric(df['age'], errors='coerce'), bins=bins, labels=labels, right=False)

print("\nDemographic Distribution - Age Groups:")
age_grp_counts = age_group.value_counts(dropna=False)
print(age_grp_counts.to_frame('count').assign(percentage=lambda x: round(x['count'] / len(df) * 100, 2)))

if 'eligibility' in df.columns:
    ct = pd.crosstab(age_group, df['eligibility'], normalize='index')
    print("\nEligibility Rate by Age Group (Demographic Representation):")
    print((ct * 100).round(2))
    if 'ELIGIBLE' in ct.columns:
        parity_diff = ct['ELIGIBLE'].max() - ct['ELIGIBLE'].min()
        print(f"\nEmpirical Demographic Parity Difference across Age Groups: {parity_diff:.4f} (Baseline: ~0.2302)")
    print(f"Verified raw DataFrame schema unchanged in memory: {df.shape}")

Target Distribution:
             count
eligibility       
INELIGIBLE     250
ELIGIBLE       250

Demographic Distribution - Sex:
     count  percentage
sex                   
M      270        54.0
F      230        46.0

Demographic Distribution - Age Groups:
                     count  percentage
age                                   
Middle Age (40-59)     301        60.2
Senior (60+)           142        28.4
Young Adult (18-39)     57        11.4
Pediatric (<18)          0         0.0

Eligibility Rate by Age Group (Demographic Representation):
eligibility          ELIGIBLE  INELIGIBLE
age                                      
Young Adult (18-39)     36.84       63.16
Middle Age (40-59)      47.51       52.49
Senior (60+)            60.56       39.44

Empirical Demographic Parity Difference across Age Groups: 0.2372 (Baseline: ~0.2302)
Verified raw DataFrame schema unchanged in memory: (500, 59)


: 

### Findings & Observations
* **Dimensionality:** 500 rows × 59 columns confirmed.
* **Target Balance:** 250 ELIGIBLE and 250 INELIGIBLE confirmed.
* **Missingness Profile:** Exported to `data/interim/data_quality/missingness_profile.csv`.
* **Demographic Fair Representation:** Documented empirical demographic parity difference across age groups (~0.2302). Protected attributes must be isolated from direct model feature weights.
* **Next Step:** Run `02_taxonomy_and_category_mapping.ipynb` to standardize socioeconomic categories.

### Completion Checklist
- [x] Raw synthetic tabular dataset loaded without modification.
- [x] Schema, dtypes, and non-null counts profiled.
- [x] `data/interim/data_quality/missingness_profile.csv` generated.
- [x] Range validations on numeric columns executed.
- [x] Demographic parity difference and protected attributes audited.